# Чтение данных

Импорт необходимых библиотек

In [1]:
import polars as pl

В Polars, помимо «жадных» (eager) методов вроде `read_csv()` и `read_parquet()`, существуют lazy аналоги — `scan_csv()` и `scan_parquet()`. Они не загружают данные сразу в память, а создают план выполнения запроса (*LazyFrame*), который будет выполнен только при вызове `.collect()`. Это позволяет оптимизатору Polars применить такие техники, как *predicate pushdown* (фильтрация на этапе чтения) и *projection pushdown* (чтение только нужных столбцов), что особенно полезно при работе с большими файлами.

### Метод `scan_csv()`

Метод `scan_csv()` позволяет лениво читать один или несколько CSV-файлов (в том числе по шаблону glob). Он возвращает объект *LazyFrame*, который можно трансформировать цепочкой операций, не загружая всё содержимое файла в память.

Основные параметры:
- `source` — путь к файлу, список путей или шаблон (например, **"data/*.csv"**). Поддерживает локальные и облачные источники (S3, GCS и др.).
- `has_header` — есть ли заголовок в файле (по умолчанию **True**).
- `separator` — символ-разделитель (по умолчанию **','**).
- `schema` / `schema_overrides` — явное указание или переопределение типов столбцов.
- `infer_schema` — автоматическое определение типов (по умолчанию **True**). Можно отключить для ускорения.
- `infer_schema_length` — сколько строк использовать для вывода схемы (по умолчанию **100**).
- `n_rows` — прочитать не более указанного числа строк.
- `skip_rows` — пропустить указанное число строк перед началом чтения.
- `low_memory` — режим с пониженным потреблением памяти (медленнее, но безопаснее для больших файлов).
- `glob` — разрешить раскрытие шаблонов вроде *.csv (по умолчанию **True**).
- `storage_options` — параметры подключения к облачным хранилищам (например, AWS S3).
- `include_file_paths` — добавить столбец с путём к исходному файлу (полезно при чтении нескольких файлов).
- `cache` — кэшировать результат после первого чтения (по умолчанию **True**).

Пример:

In [2]:
lf = pl.scan_csv("btcusd_1-min_data.csv")\
    .with_columns(pl.col("Timestamp").cast(pl.Datetime).dt.with_time_unit("ms"))\
    .filter(pl.col("Volume") > 100)\
    .select(["Timestamp", "Close"])\
    .group_by(pl.col("Timestamp").dt.date().alias("date")).agg(pl.col("Close").mean())\
    .sort("date")\
    .collect()

lf

C:\Users\ardat\AppData\Local\Temp\ipykernel_27060\2139474968.py:2: DeprecationWarning: `ExprDateTimeNameSpace.with_time_unit` is deprecated. Instead, first cast to `Int64` and then cast to the desired data type.
  .with_columns(pl.col("Timestamp").cast(pl.Datetime).dt.with_time_unit("ms"))\


date,Close
date,f64
1970-01-16,84.992163
1970-01-17,427.279519
1970-01-18,5210.358705
1970-01-19,11981.692965
1970-01-20,30549.552665
1970-01-21,92788.3125


В этом примере Polars прочитает только нужные столбцы и пропустит строки с объёмом ≤ 100 ещё на этапе парсинга CSV, что экономит и время, и память.

### Метод pl.scan_parquet()

Метод `scan_parquet()` - ленивая версия чтения Parquet-файлов. Напомню. что Parquet - это столбцовый бинарный формат, оптимизированный для аналитики. Благодаря метаданным и статистике внутри файла, Polars может пропускать целые блоки данных, не удовлетворяющие фильтрам.

Основные параметры:
- `source` — путь к файлу, директории или шаблону (например, **"s3://bucket/data/*.parquet"**).
- `n_rows` — ограничение на число читаемых строк.
- `parallel` — стратегия параллелизма:
  - `'auto'` — автоматический выбор (по умолчанию),
  - `'row_groups'` — параллельное чтение групп строк,
  - `'prefiltered'` — сначала применяет фильтры, потом читает только нужные строки (может ускорить работу при сильной фильтрации).
- `use_statistics` — использовать статистику Parquet для пропуска ненужных блоков (по умолчанию **True**).
- `hive_partitioning` — автоматически извлекать значения из путей в стиле Hive (например, **year=2024/month=05/...**).
- `schema` — явное задание схемы.
- `missing_columns` — поведение при отсутствии столбца в файле: 'raise' (ошибка) или 'insert' (добавить NULL-столбец).
- `low_memory` — режим экономии памяти.
- `storage_options` — параметры для облачных провайдеров (AWS, GCP, Azure и др.).
- `include_file_paths` — добавить столбец с путём к файлу.
- `cache` — кэшировать результат после первого чтения.

Пример:

In [3]:
lf = pl.scan_parquet("all_stock_data.parquet")\
       .filter((pl.col("Date") >= pl.date(2020, 1, 1)) & (pl.col("Volume") > 1000))\
       .select(["Ticker", "Close"])\
       .group_by("Ticker").agg(pl.col("Close").mean())\
       .sort("Close", descending=True).limit(10)\
       .collect()

lf

Ticker,Close
str,f64
"""BRK-A""",503114.572078
"""FNMFO""",9800.0
"""ADTX""",5773.984989
"""NVR""",5323.317548
"""FFIE""",4567.484561
"""GPUS""",3667.532321
"""PIXY""",3608.418304
"""UVXY""",3434.946977
"""SEB""",3403.351252


Здесь Polars:
- не загружает весь файл в память,
- использует статистику Parquet, чтобы пропустить файлы/блоки до 2020 года,
- читает только столбцы Date, Volume, Ticker, Close.